# 02 — Feature analysis

> This system predicts glucose values for research and analysis. It is not a medical device and must not be used to determine insulin doses or make treatment decisions.

Features are built by `python -m src.features` (see `reports/feature_report.md` for the catalog and leakage rules).

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd())
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src import config, db, train
settings, mcfg = config.settings(), config.models()
duck = db.duck(read_only=True)
q = lambda sql: duck.execute(sql).df()

In [ ]:
df, catalog = train.load_dataset(settings)
usable = df[df.current_glucose.notna() & df.insulin_known]
print(df.shape, len(usable))
catalog.groupby('group').size()

## Missingness among usable samples

In [ ]:
miss = usable[catalog.feature].isna().mean().sort_values(ascending=False) * 100
miss.head(15).round(2)

## Correlation with the 30-minute change

Simple correlations only: they ignore interactions and are not causal.

In [ ]:
corr = usable[catalog.feature.tolist() + ['delta_30m']].corr(numeric_only=True)['delta_30m'].drop('delta_30m')
corr.reindex(corr.abs().sort_values(ascending=False).index).head(20).round(3)

## Estimated IOB versus recent insulin

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
s = usable.sample(5000, random_state=0)
ax.scatter(s.insulin_last_120m, s.estimated_iob, s=3, alpha=0.3)
ax.set_xlabel('insulin last 120 min (U)'); ax.set_ylabel('estimated IOB (U, approximation)')
print('correlation:', round(usable[['insulin_last_120m', 'estimated_iob']].corr().iloc[0, 1], 3))